Proyecto Final

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import Window

# Crear la sesión de Spark
spark = SparkSession.builder \
    .appName("Pract2_PySpark") \
    .getOrCreate()

# Verificar la versión
print("Versión de Spark:", spark.version)

sc = spark.sparkContext

DATA_PATH = "/home/FinPlus/work/data/"

Versión de Spark: 3.5.0


Impotamos la data

In [3]:
client = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'CLIENTS.csv'))
client.show(5, 0)

+------------+----------------------+-----------------+------+------------+--------------+-----------+--------------+--------------+--------------------+------------------+------------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|CLIENT_ID   |NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION     |MARITAL_STATUS|HOME_SITUATION      |REGION_SC

In [ ]:
beh = (spark.read.option('header', 'true').option('delimiter',',')
                     .csv(DATA_PATH + 'BEHAVIOURAL.csv'))
beh.show(5, 0)

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|CONTRACT_ID       |CLIENT_ID   |DATE      |CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821000018d00XXX|ES182394447V|2021-08-29|491.21              |540.0            |0.0                     |28.03               |28.03                   |0.0                       |46.81              |0

In [ ]:
client_sin_duplicados_por_id = client.dropDuplicates(['CLIENT_ID'])

client_sin_duplicados_por_id.show(5)

print(f"Filas originales en client: {client.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en client: {client_sin_duplicados_por_id.count()}")

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+------------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+
|   CLIENT_ID|NON_COMPLIANT_CONTRACT|NAME_PRODUCT_TYPE|GENDER|TOTAL_INCOME|AMOUNT_PRODUCT|INSTALLMENT|EDUCATION|MARITAL_STATUS|HOME_SITUATION|REGION_SCORE|     AGE_IN_YEARS|JOB_SEN

In [ ]:
beh_sin_duplicados_por_id = beh.dropDuplicates(['CLIENT_ID', 'DATE', 'CREDICT_CARD_BALANCE', 'CONTRACT_ID'])

beh_sin_duplicados_por_id.show(5)

print(f"Filas originales en beh: {beh.count()}")
print(f"Filas después de eliminar duplicados (por CLIENT_ID) en beh: {beh_sin_duplicados_por_id.count()}")

+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|       CONTRACT_ID|   CLIENT_ID|      DATE|CREDICT_CARD_BALANCE|CREDIT_CARD_LIMIT|CREDIT_CARD_DRAWINGS_ATM|CREDIT_CARD_DRAWINGS|CREDIT_CARD_DRAWINGS_POS|CREDIT_CARD_DRAWINGS_OTHER|CREDIT_CARD_PAYMENT|NUMBER_DRAWINGS_ATM|NUMBER_DRAWINGS|NUMBER_INSTALMENTS|CURRENCY|
+------------------+------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+-------------------+-------------------+---------------+------------------+--------+
|ES1821489396v00XXX|ES182100006A|2021-08-29|                 0.0|           3240.0|                     0.0|                 0.0|                     0.0|                       0.0|                0.0| 

No hay filas duplicadas en todo el data set tanto en Clientes como en Behavioural.

In [ ]:
for column in client.columns:
    null_count = client.filter(client[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")

Columna 'CLIENT_ID': 0 valores nulos
Columna 'NON_COMPLIANT_CONTRACT': 0 valores nulos
Columna 'NAME_PRODUCT_TYPE': 0 valores nulos
Columna 'GENDER': 0 valores nulos
Columna 'TOTAL_INCOME': 0 valores nulos
Columna 'AMOUNT_PRODUCT': 0 valores nulos
Columna 'INSTALLMENT': 7 valores nulos
Columna 'EDUCATION': 39640 valores nulos
Columna 'MARITAL_STATUS': 2 valores nulos
Columna 'HOME_SITUATION': 0 valores nulos
Columna 'REGION_SCORE': 0 valores nulos
Columna 'AGE_IN_YEARS': 0 valores nulos
Columna 'JOB_SENIORITY': 29174 valores nulos
Columna 'HOME_SENIORITY': 0 valores nulos
Columna 'LAST_UPDATE': 0 valores nulos
Columna 'OWN_INSURANCE_CAR': 0 valores nulos
Columna 'CAR_AGE': 107550 valores nulos
Columna 'FAMILY_SIZE': 2 valores nulos
Columna 'REACTIVE_SCORING': 91901 valores nulos
Columna 'PROACTIVE_SCORING': 337 valores nulos
Columna 'BEHAVIORAL_SCORING': 32246 valores nulos
Columna 'DAYS_LAST_INFO_CHANGE': 1 valores nulos
Columna 'NUMBER_OF_PRODUCTS': 21903 valores nulos
Columna 'OCCUP

In [ ]:
for column in beh.columns:
    null_count = beh.filter(beh[column].isNull()).count()
    print(f"Columna '{column}': {null_count} valores nulos")

Columna 'CONTRACT_ID': 0 valores nulos
Columna 'CLIENT_ID': 0 valores nulos
Columna 'DATE': 0 valores nulos
Columna 'CREDICT_CARD_BALANCE': 0 valores nulos
Columna 'CREDIT_CARD_LIMIT': 0 valores nulos
Columna 'CREDIT_CARD_DRAWINGS_ATM': 0 valores nulos
Columna 'CREDIT_CARD_DRAWINGS': 0 valores nulos
Columna 'CREDIT_CARD_DRAWINGS_POS': 0 valores nulos
Columna 'CREDIT_CARD_DRAWINGS_OTHER': 0 valores nulos
Columna 'CREDIT_CARD_PAYMENT': 0 valores nulos
Columna 'NUMBER_DRAWINGS_ATM': 0 valores nulos
Columna 'NUMBER_DRAWINGS': 0 valores nulos
Columna 'NUMBER_INSTALMENTS': 0 valores nulos
Columna 'CURRENCY': 0 valores nulos


In [ ]:
client.printSchema()

root
 |-- CLIENT_ID: string (nullable = true)
 |-- NON_COMPLIANT_CONTRACT: string (nullable = true)
 |-- NAME_PRODUCT_TYPE: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- TOTAL_INCOME: string (nullable = true)
 |-- AMOUNT_PRODUCT: string (nullable = true)
 |-- INSTALLMENT: string (nullable = true)
 |-- EDUCATION: string (nullable = true)
 |-- MARITAL_STATUS: string (nullable = true)
 |-- HOME_SITUATION: string (nullable = true)
 |-- REGION_SCORE: string (nullable = true)
 |-- AGE_IN_YEARS: string (nullable = true)
 |-- JOB_SENIORITY: string (nullable = true)
 |-- HOME_SENIORITY: string (nullable = true)
 |-- LAST_UPDATE: string (nullable = true)
 |-- OWN_INSURANCE_CAR: string (nullable = true)
 |-- CAR_AGE: string (nullable = true)
 |-- FAMILY_SIZE: string (nullable = true)
 |-- REACTIVE_SCORING: string (nullable = true)
 |-- PROACTIVE_SCORING: string (nullable = true)
 |-- BEHAVIORAL_SCORING: string (nullable = true)
 |-- DAYS_LAST_INFO_CHANGE: string (nullable = 

In [ ]:
beh.printSchema()

root
 |-- CONTRACT_ID: string (nullable = true)
 |-- CLIENT_ID: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- CREDICT_CARD_BALANCE: string (nullable = true)
 |-- CREDIT_CARD_LIMIT: string (nullable = true)
 |-- CREDIT_CARD_DRAWINGS_ATM: string (nullable = true)
 |-- CREDIT_CARD_DRAWINGS: string (nullable = true)
 |-- CREDIT_CARD_DRAWINGS_POS: string (nullable = true)
 |-- CREDIT_CARD_DRAWINGS_OTHER: string (nullable = true)
 |-- CREDIT_CARD_PAYMENT: string (nullable = true)
 |-- NUMBER_DRAWINGS_ATM: string (nullable = true)
 |-- NUMBER_DRAWINGS: string (nullable = true)
 |-- NUMBER_INSTALMENTS: string (nullable = true)
 |-- CURRENCY: string (nullable = true)



In [ ]:
client.columns

['CLIENT_ID',
 'NON_COMPLIANT_CONTRACT',
 'NAME_PRODUCT_TYPE',
 'GENDER',
 'TOTAL_INCOME',
 'AMOUNT_PRODUCT',
 'INSTALLMENT',
 'EDUCATION',
 'MARITAL_STATUS',
 'HOME_SITUATION',
 'REGION_SCORE',
 'AGE_IN_YEARS',
 'JOB_SENIORITY',
 'HOME_SENIORITY',
 'LAST_UPDATE',
 'OWN_INSURANCE_CAR',
 'CAR_AGE',
 'FAMILY_SIZE',
 'REACTIVE_SCORING',
 'PROACTIVE_SCORING',
 'BEHAVIORAL_SCORING',
 'DAYS_LAST_INFO_CHANGE',
 'NUMBER_OF_PRODUCTS',
 'OCCUPATION',
 'DIGITAL_CLIENT',
 'HOME_OWNER',
 'EMPLOYER_ORGANIZATION_TYPE',
 'CURRENCY',
 'NUM_PREVIOUS_LOAN_APP',
 'LOAN_ANNUITY_PAYMENT_MAX',
 'LOAN_ANNUITY_PAYMENT_MIN',
 'LOAN_ANNUITY_PAYMENT_SUM',
 'LOAN_APPLICATION_AMOUNT_MAX',
 'LOAN_APPLICATION_AMOUNT_MIN',
 'LOAN_APPLICATION_AMOUNT_SUM',
 'LOAN_CREDIT_GRANTED_MAX',
 'LOAN_CREDIT_GRANTED_MIN',
 'LOAN_CREDIT_GRANTED_SUM',
 'LOAN_VARIABLE_RATE_MAX',
 'LOAN_VARIABLE_RATE_MIN',
 'NUM_STATUS_ANNULLED',
 'NUM_STATUS_AUTHORIZED',
 'NUM_STATUS_DENIED',
 'NUM_STATUS_NOT_USED',
 'NUM_FLAG_INSURED']

In [ ]:
beh.columns

['CONTRACT_ID',
 'CLIENT_ID',
 'DATE',
 'CREDICT_CARD_BALANCE',
 'CREDIT_CARD_LIMIT',
 'CREDIT_CARD_DRAWINGS_ATM',
 'CREDIT_CARD_DRAWINGS',
 'CREDIT_CARD_DRAWINGS_POS',
 'CREDIT_CARD_DRAWINGS_OTHER',
 'CREDIT_CARD_PAYMENT',
 'NUMBER_DRAWINGS_ATM',
 'NUMBER_DRAWINGS',
 'NUMBER_INSTALMENTS',
 'CURRENCY']

In [ ]:
df = client.join(beh, "CLIENT_ID")
df.show()

+------------+----------------------+-----------------+------+------------+--------------+-----------+---------+--------------+--------------+------------+-----------------+-------------+--------------+-----------+-----------------+-------+-----------+----------------+------------------+------------------+---------------------+------------------+----------+--------------+----------+--------------------------+--------+---------------------+------------------------+------------------------+------------------------+---------------------------+---------------------------+---------------------------+-----------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------+---------------------+-----------------+-------------------+----------------+------------------+----------+--------------------+-----------------+------------------------+--------------------+------------------------+--------------------------+----------------

In [ ]:
(df.groupBy('NAME_PRODUCT_TYPE')
    .count()
    .orderBy(F.desc('count'))
).show(10)

+-----------------+-------+
|NAME_PRODUCT_TYPE|  count|
+-----------------+-------+
|        PRODUCT 1|1696040|
|        PRODUCT 2|  28814|
+-----------------+-------+



In [ ]:
(df.groupBy('TOTAL_INCOME')
    .count()
    .orderBy(F.desc('count'))
).show(10)

+------------+------+
|TOTAL_INCOME| count|
+------------+------+
|      1620.0|192695|
|      1890.0|155320|
|      1350.0|151377|
|      2160.0|150316|
|      2700.0|134819|
|      1080.0|107596|
|      2430.0|102148|
|      3240.0| 75844|
|      3780.0| 42788|
|       810.0| 42044|
+------------+------+
only showing top 10 rows

